# Probability maps animation

This notebook creates an animation showing the evolution of the predicted time-dependent deposit probability from 170Ma-present. The notebook `02-create_probability_maps.ipynb` must have been run previously.

If training data has been extracted from the source datasets by running the `00c-extract_training_data_global.ipynb` and `00b-extract_grid_data.ipynb` notebooks, set the `use_extracted_data` variable below to `True` to use this dataset instead of the pre-prepared training data from the [Zenodo repository](https://zenodo.org/record/8157691).

## Notebook setup

These cells set some of the important variables and definitions used throughout the notebook.

### Config

In [ ]:
config_file = "config/.run_config.yml"

In [ ]:
from lib.paths import PathConfigManager

pcm = PathConfigManager(config_file, notebook="03")

# =====================
# Filestructure
# =====================

plate_model_dir = pcm.PLATE_MODEL_DIR
output_dir = pcm.OUTPUT_DIR

pcm.create_directories()

# =====================
# Plate model
# =====================

plate_model_name = pcm.config["plate_model"]["plate_model_name"]
use_provided_plate_model = pcm.use_provided_plate_model

min_time = pcm.config["timespan"]["min"]
max_time = pcm.config["timespan"]["max"]
times = range(min_time, max_time + 1)

# =====================
# Notebook scope
# =====================

use_extracted_data = pcm.use_extracted_data

# =====================
# Run parameters
# =====================

n_jobs = pcm.config["n_jobs"]
overwrite = pcm.config["overwrite_output"]
verbose = pcm.config["verbose"]



### Imports

In [ ]:
import tempfile
import warnings
from itertools import product
from pathlib import Path

import cartopy.crs as ccrs
import pandas as pd
import pygplates
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately import (
        PlateReconstruction,
        PlotTopologies,
    )
from joblib import Parallel, delayed

from lib.animation import create_animation
from lib.check_files import (
    check_plate_model,
    check_prepared_data,
)
from lib.misc import (
    filter_topological_features,
    reconstruct_by_topologies,
)
from lib.plate_models import (
    get_plate_reconstruction,
    get_plot_topologies,
)
from lib.visualisation import plot

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning

NOTEBOOK = "get_ipython" in dir()

env: PYTHONWARNINGS=ignore::UserWarning


### Input and output files

If necessary, the plate model will be downloaded:

In [ ]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model, _tf = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
    filter_topologies=True,
)

if use_provided_plate_model:
    coastlines_filenames = [
        str(
            plate_model_dir
            / "StaticGeometries"
            / "AgeGridInput"
            / "CombinedTerranes.gpml"
        )
    ]
    gplot = PlotTopologies(
        plate_model,
        coastlines=coastlines_filenames,
    )
else:
    gplot = get_plot_topologies(
        model_name=plate_model_name,
        model_dir=plate_model_dir,
        plate_reconstruction=plate_model,
        filter_topologies=True,
    )

if use_extracted_data:
    training_filename = pcm.TRAINING_DATA_PATH
    if verbose:
        print(f"Using extracted training data: {training_filename}")
else:
    prep_dir = check_prepared_data("prepared_data", verbose=True)
    prep_dir = Path(prep_dir)
    training_filename = prep_dir / "training_data_global.csv"

### Plot options

These should generally be left unchanged.

In [6]:
lon_0 = 0.0
projection = ccrs.Mollweide(lon_0)

imshow_kwargs = dict(
    vmin=0,
    vmax=60,
    cmap="viridis"
)

### Load training data

The training dataset is used to plot deposit locations on the maps.

In [7]:
training_data = pd.read_csv(training_filename)
positives = training_data[training_data["label"] == "positive"]

### Reconstruct deposit data for plots

In [8]:
positives = reconstruct_by_topologies(
    data=(positives[["lon", "lat", "age (Ma)", "label", "region"]]).copy(),
    plate_reconstruction=plate_model,
    times=times,
)

## Example plot

Create an example plot at a single time step (by default, 110 Ma).

In [ ]:
# Example plot (only in notebook)
if NOTEBOOK:
    t_example = 110  # Ma
    gplot.time = t_example
    probs_dir = pcm.PROBABILITY_GRIDS_DIR

    print("PU example plot:")
    fig = plot(
        gplot=gplot,
        probabilities=str(
            probs_dir / f"probability_grid_{t_example:0.0f}Ma.nc"
        ),
        positives=positives,
        projection=projection,
        time=t_example,
        central_meridian=lon_0,
        imshow_kwargs=imshow_kwargs,
    )

## Create all plots

This will create the plots for all models.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir, \
        Parallel(
            n_jobs=n_jobs,
            verbose=int(verbose),
            pre_dispatch="all",
            batch_size=int(len(times) // n_jobs) + 1,
        ) as parallel:

    # Create animation for global model
    output_filename = pcm.PROBABILITY_OUTPUT_DIR / "probability_animation.mp4"
    probs_dir = pcm.PROBABILITY_GRIDS_DIR
    if not probs_dir.is_dir():
        pass
    else:
        # Create all plots
        tmpdir_path = Path(tmpdir)
        output_filenames = [
            str(tmpdir_path / f"image_{t:0.0f}Ma.png")
            for t in times
        ]
        parallel(
            delayed(plot)(
                gplot=gplot,
                probabilities=str(
                    probs_dir / f"probability_grid_{t:0.0f}Ma.nc"
                ),
                projection=projection,
                time=t,
                positives=positives,
                output_filename=o,
                central_meridian=lon_0,
                imshow_kwargs=imshow_kwargs,
            )
            for t, o in zip(times, output_filenames)
        )
        create_animation(
            image_filenames=output_filenames[::-1],  # reverse order of frames (forward in time)
            output_filename=str(output_filename),
            fps=10,  # framerate of output video
            bitrate="5000k",
        )

    # Create animations for regional models
    for region, positives_region in positives.groupby("region"):
        r = "_".join(region.lower().split())
        probs_dir_region = pcm.PROBABILITY_OUTPUT_DIR / f"probability_grids_{r}"
        if not probs_dir_region.is_dir():
            continue
        output_filename_region = pcm.PROBABILITY_OUTPUT_DIR / f"probability_animation_{r}.mp4"
        parallel(
            delayed(plot)(
                gplot=gplot,
                probabilities=str(
                    probs_dir_region
                    / f"probability_grid_{t:0.0f}Ma.nc"
                ),
                projection=projection,
                time=t,
                positives=positives_region,
                output_filename=o,
                central_meridian=lon_0,
                imshow_kwargs=imshow_kwargs,
            )
            for t, o in zip(times, output_filenames)
        )
        create_animation(
            image_filenames=output_filenames[::-1],
            output_filename=str(output_filename_region),
            fps=10,
            bitrate="5000k",
        )